# Цифровые технологии в профессиональной деятельности
## Раздел 1. Текст-майнинг
## Семинар 4. Лабораторная работа: TF-IDF на конституциях

На предыдущих семинарах мы закрепили препроцессинг текста, научились рассматривать показатели частотности, заносить их в датафреймы. Одну из метрик — TF-IDF — вы разберёте самостоятельно в ходе лабораторной работы, а заодно в очередной раз повторите шаги работы с текстовыми данными.

**Материал:** шесть основных законов российского государства — от Свода основных государственных законов 1906 года до Конституции РФ 1993 года.

**Задача:** с помощью TF-IDF выделить ключевые слова каждого периода и подумать, что они говорят о языке власти.

**План:**
1. Формируем корпус текстов.
2. Препроцессинг: токенизация, лемматизация.
3. Считаем TF-IDF — выделяем ключевые слова.
4. Делаем выводы.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
import pymorphy3
from razdel import tokenize
from collections import Counter
from nltk.corpus import stopwords
import nltk
from wordcloud import WordCloud

nltk.download('stopwords', quiet=True)
morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words('russian'))

---
## Часть 1. Формирование корпуса

In [ ]:
# Загрузка текстов
import requests

base_url = "https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/04_labwork"

filenames = {
    '1906': '1906.txt',
    '1918': '1918.txt',
    '1924': '1924.txt',
    '1936': '1936.txt',
    '1977': '1977.txt',
    '1993': '1993.txt'
}

corpus = {}
for year, fname in filenames.items():
    response = requests.get(f"{base_url}/{fname}")
    response.encoding = 'utf-8'
    corpus[year] = response.text

print(f"Загружено {len(corpus)} текстов")

#### Задание 1
Какой текст самый длинный? Какой самый короткий? Выведите длины с помощью кода.

Ответ:

In [ ]:
# ВАШ КОД

Как вы думаете, может ли разница в длине повлиять на результаты анализа частотности?

---
## Часть 2. Препроцессинг

#### Задание 2
Функция ниже выполняет токенизацию и лемматизацию. В ней пропущены две строчки — допишите их.

In [ ]:
def preprocess(text):
    """Токенизирует, лемматизирует текст и убирает стоп-слова."""
    # Токенизация: разбиваем текст на слова с помощью razdel
    tokens = [ # ВАШ КОД ]
    
    lemmas = []
    for word in tokens:
        # --- ДОПИШИТЕ: получите лемму с помощью pymorphy3 ---
        lemma = ...  # ВАШ КОД
        
        # --- ДОПИШИТЕ: добавьте условие фильтрации ---
        if ...:  # ВАШ КОД: лемма не в стоп-словах И длина > 3
            lemmas.append(lemma)
    
    return lemmas

Когда допишете функцию, допишите ячейку ниже — она применит препроцессинг ко всему корпусу.

In [ ]:
# Применяем препроцессинг
texts_by_year = {}
counts_by_year = {}

for year, text in corpus.items():
    lemmas = preprocess(text)
    texts_by_year[year] = lemmas
    counts_by_year[year] = Counter(lemmas)

# Создаём DataFrame, заполняем пустые значения нулями
df = # ВАШ КОД
print(f"Таблица: {df.shape[0]} лемм × {df.shape[1]} документов")
df.head()

Какого размера получилась ваша таблица? Отправляйте результат в чат.

---
## Часть 3. TF-IDF

### Зачем нужен TF-IDF, если у нас есть частоты?

Cамые частотные слова нередко оказываются общими для всех текстов: «государство», «развитие», «страна». Их абсолютная или относительная частота будут довольно высокими на нашем материале, но это не помогает точно определить ключевые понятия периода. Если в случае больших разнородных корпусов хватит даже абсолютной частотности, то в корпусе с однородными текстами найти ключевые слова труднее.

**TF-IDF (Term Frequency — Inverse Document Frequency)** решает эту проблему. Метрика повышает вес слов, которые часты в конкретном тексте, но редки в коллекции в целом.

Разберём метрику по шагам.

### Шаг 1. Term Frequency (TF) — частота слова в документе

TF отвечает на вопрос: **насколько часто слово встречается в данном тексте?**

$$\text{TF}(t, d) = \frac{\text{Число вхождений слова } t \text{ в документ } d}{\text{Общее число слов в документе } d}$$

По сути, это относительная частота, которую мы уже считали на втором семинаре, только нормированная на единицу, а не на тысячу.

Посчитаем TF для слова «Россия» в Конституциях.

In [ ]:
# TF для слова «Россия»

word = 'Россия'

for year in df.columns:
    # Количество вхождений слова в данный год
    count = df.loc[word, year] if word in df.index else 0
    
    # Общее число слов (лемм) в данный год
    total = df[year].sum()
    
    # TF = доля слова в тексте
    tf = count / total
    
    print(f'{year}: слово «{word}» встретилось {count} раз из {total} слов\n')
    print(f'       TF = {count}/{total} = {tf:.6f}')
    print()

Выберите слово вам по нраву, посчитайте его TF, отправьте результат в чат.

### Шаг 2. Inverse Document Frequency (IDF) — обратная документная частота

IDF отвечает на вопрос: **насколько слово уникально для коллекции?**

$$\text{IDF}(t) = \log\frac{N}{\text{DF}(t)}$$

где $N$ — общее число документов в коллекции, а $\text{DF}(t)$ — в скольких документах встречается слово $t$.

Идея: если слово встречается **во всех** документах (например, «государство»), то $\text{DF} = N$, и $\text{IDF} = \log(1) = 0$. Такое слово получит нулевой вес — оно не помогает различать тексты.

Если слово встречается **только в одном** документе, то $\text{IDF} = \log(N)$ — максимальный вес.

Функция логарифма здесь нужна, чтобы сгладить различия: без неё слово из одного документа получило бы вес в 8 раз больше, чем слово из всех документов. Логарифм делает шкалу более плавной.

In [ ]:
import math

N = len(df.columns)  # число документов
print(f'Всего документов: {N}')
print()

for word in ['государство', 'россия', 'революция', 'советский', 'колхоз']:
    if word not in df.index:
        print(f'«{word}»: не найдено в корпусе')
        continue
    
    # DF = в скольких годах слово встречается хотя бы раз
    document_freq = (df.loc[word] > 0).sum()
    
    # IDF = log(N / DF)
    idf = math.log(N / document_freq)
    
    print(f'«{word}»: встречается в {document_freq} из {N} документов\n')
    print(f'         IDF = log( {N}/{document_freq} ) = {idf:.4f}')
    print()

Выберите слово вам по нраву, посчитайте его IDF, отправьте результат в чат.

### Шаг 3. TF-IDF = TF × IDF

Теперь перемножаем. Слово получает высокий вес, если оно одновременно:
- **часто** в данном документе (высокий TF)
- **редко** в коллекции в целом (высокий IDF)

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Посчитаем TF-IDF для нескольких слов и сравним.

#### Задание 3

In [ ]:
# Считаем TF-IDF вручную для нескольких слов

test_words = ['демократия', 'свобода', 'безопасность', 'коммунизм']

print(f"{'Слово':<16} {'Год':<6} {'TF':>10} {'IDF':>8} {'TF-IDF':>10}")
print('-' * 55)

for word in test_words:
    if word not in df.index:
        continue

    # document_freq =
    # idf =


    for year in df.columns:

        # ВАШ КОД

        print(f'{word:<16} {year:<6} {tf:>10.6f} {idf:>8.4f} {tfidf:>10.6f}')

    print()
    
    # ВАШ КОД:
    # 1. Посчитайте document_freq и idf (как в шаге 2)
    # 2. Пройдитесь циклом по df.columns (годам)
    # 3. Для каждого года посчитайте tf и tfidf
    # 4. Выведите результат через print


### Готовое решение: sklearn

Изучаемые нами метрики уже реализованы в специальных библиотеках. Мы пишем код самостоятельно, чтобы понять, как они работают, но на практике удобнее использовать готовые решения. `TfidfVectorizer` из библиотеки `sklearn` делает то же самое (с некоторыми дополнительными нормализациями), но быстрее и надёжнее.

Обратите внимание: `TfidfVectorizer` принимает на вход **строки**, а не списки слов. Поэтому нам нужно склеить леммы обратно в строки.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Подготовим тексты: список лемм -> строка
corpus_texts = {}
for year, lemmas in texts_by_year.items():
    corpus_texts[year] = ' '.join(lemmas)

# Посмотрим, как выглядит результат для одного года
for key, value in corpus_texts.items():
    print(f'{key}: {value[:100]}...')
    break

years_list = sorted(corpus_texts.keys())
texts_list = [corpus_texts[y] for y in years_list]

# Считаем TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(texts_list)

feature_names = vectorizer.get_feature_names_out()
print(f'\nРазмер матрицы: {tfidf_matrix.shape}')
print(f'(строки = годы, столбцы = уникальные слова)')

In [ ]:
# Извлекаем топ-10 слов с наивысшим TF-IDF для каждого года
for i, year in enumerate(years_list):
    # Берём строку матрицы для данного года и превращаем в обычный массив
    row = tfidf_matrix[i].toarray().flatten()

    # Сортируем индексы по убыванию значения TF-IDF, берём первые 10
    top_indices = row.argsort()[-10:][::-1]

    # Извлекаем слова и их веса
    top_words = [(feature_names[j], round(row[j], 3)) for j in top_indices]
    
    print(f'\n{year}:')
    for word, score in top_words:
        print(f'  {word}: {score}')

### Визуализация

Облака слов для каждого документа — готовая функция, просто запустите.

In [ ]:
def plot_wordclouds(matrix, feature_names, labels, ncols=3):
    """Рисует облака слов для каждого документа."""
    n = len(labels)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, label in enumerate(labels):
        row = matrix[i].toarray().flatten()
        word_scores = {feature_names[j]: row[j] for j in range(len(row)) if row[j] > 0}
        
        wc = WordCloud(width=600, height=400, background_color='white',
                       colormap='viridis', max_words=40)
        wc.generate_from_frequencies(word_scores)
        
        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(label, fontsize=16, fontweight='bold')
        axes[i].axis('off')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

plot_wordclouds(tfidf_matrix, feature_names, years_list)

---
## Часть 4. Анализ и выводы

#### Задание 4
Рассмотрите ключевые слова каждого документа. Ответьте на вопросы (2–3 предложения на каждый):

1. Какие слова-маркеры выделяют каждую конституцию? Соотнесите их с историческим контекстом.

2. Есть ли слова, которые присутствуют в топ-10 **нескольких** конституций? О чём это говорит?

3. Что TF-IDF может и чего **не может** сказать о реальности за текстом?

In [ ]:
# Место для дополнительного кода


- Место для ваших заметок

#### Задание 5
У вас есть несколько текстов конституций, рассчитанный TF-IDF, ключевые слова из каждого текста. С одной стороны некоторые из них являются реалиями, указывающими на определённый временной промежуток, с другой – маркерами стиля текста. **Являются ли в нашем случае ключевые слова решающими показателями стиля? В каких случаях ключевые слова совсем не помогут в определении стиля или при авторской атрибуции?**

Запишите ответ — мы вернёмся к этому вопросу при разборе стилометрии.

Ответ: 

---
### Итоги

В этой лабораторной мы:
1. Повторили препроцессинг текста на новом материале
2. Применили TF-IDF для выделения ключевых слов российских конституций
3. Увидели, как менялся язык государственной власти за 90 лет
4. Осознали ограничение метода